In [2]:
import pandas as pd
import numpy as np

print("Cargando Rayos X del Dataset Crudo...")
df = pd.read_parquet("data/raw/sima_raw.parquet")

cols_medicion = ['CO', 'NO', 'NO2', 'NOX', 'O3', 'PM10', 'PM2.5', 'PRS', 'RAINF', 'RH', 'SO2', 'SR', 'TOUT', 'WSR', 'WDR']

# Creamos una copia de trabajo temporal. 
# Al forzar pd.to_numeric con 'coerce', las letras de error se vuelven NaN para poder contarlas.
df_check = df.copy()
for col in cols_medicion:
    df_check[col] = pd.to_numeric(df_check[col], errors='coerce')

df_check['Year'] = df_check['Date'].dt.year

print("\n=======================================================")
print("📊 REPORTE 1: PORCENTAJE DE DATOS PERDIDOS POR AÑO")
print("=======================================================")
missing_por_ano = df_check.groupby('Year')[cols_medicion].apply(lambda x: x.isna().mean() * 100).round(2)
display(missing_por_ano)

print("\n=======================================================")
print("📊 REPORTE 2: SALUD DE LOS SENSORES POR ESTACIÓN (% Perdido)")
print("=======================================================")
missing_por_estacion = df_check.groupby('Estacion')[cols_medicion].apply(lambda x: x.isna().mean() * 100).round(2)
display(missing_por_estacion)

print("\n=======================================================")
print("📊 REPORTE 3: RACHAS MÁXIMAS SIN DATOS (ESTACIÓN 'CE')")
print("Para detectar si un sensor mide diario o estuvo apagado meses")
print("=======================================================")

def max_consecutive_nans(series):
    is_nan = series.isna()
    if not is_nan.any():
        return 0
    # Agrupa los NaNs consecutivos y suma cuántos hay en cada racha
    consecutive = is_nan.groupby((~is_nan).cumsum()).sum()
    return int(consecutive.max())

# Analizamos solo la estación Centro (CE) como muestra rápida
df_ce = df_check[df_check['Estacion'] == 'CE'].sort_values('Date')
rachas = {col: max_consecutive_nans(df_ce[col]) for col in cols_medicion}

df_rachas = pd.DataFrame(list(rachas.items()), columns=['Contaminante', 'Max_Horas_Consecutivas_Vacias'])
df_rachas['Dias_Equivalentes'] = (df_rachas['Max_Horas_Consecutivas_Vacias'] / 24).round(1)

display(df_rachas.sort_values('Max_Horas_Consecutivas_Vacias', ascending=False))

Cargando Rayos X del Dataset Crudo...

📊 REPORTE 1: PORCENTAJE DE DATOS PERDIDOS POR AÑO


,CO,NO,NO2,NOX,O3,PM10,PM2.5,PRS,RAINF,RH,SO2,SR,TOUT,WSR,WDR
Year,,,,,,,,,,,,,,,
2020,43.65,61.10,69.54,65.64,60.57,7.55,23.86,12.86,14.45,17.25,44.28,1.88,11.14,19.93,30.88
2021,8.95,15.82,15.46,15.90,13.05,5.73,18.05,5.77,5.83,7.28,19.17,1.82,6.95,11.53,9.89
2022,3.45,4.13,3.52,3.50,3.65,3.19,17.07,2.70,1.66,3.85,8.85,3.16,2.11,2.61,4.66
2023,4.67,4.15,3.31,3.33,4.96,3.08,26.10,1.70,1.25,10.82,4.05,4.54,1.55,3.09,2.11
2024,8.72,5.48,5.09,4.75,4.85,3.76,31.01,3.03,1.89,10.99,8.13,5.50,8.35,2.28,2.15
2025,13.54,14.84,13.24,13.39,6.59,4.83,34.10,5.35,4.53,18.13,9.50,4.65,11.83,3.57,3.28



📊 REPORTE 2: SALUD DE LOS SENSORES POR ESTACIÓN (% Perdido)


,CO,NO,NO2,NOX,O3,PM10,PM2.5,PRS,RAINF,RH,SO2,SR,TOUT,WSR,WDR
Estacion,,,,,,,,,,,,,,,
CE,5.78,9.08,6.78,7.01,7.33,2.79,14.15,2.04,1.91,2.34,4.83,1.26,1.95,2.14,2.00
NE,11.50,15.07,22.18,22.18,23.10,3.56,7.17,5.07,4.17,5.51,22.06,2.64,26.87,13.38,7.76
NE2,15.16,27.62,27.57,26.70,21.98,5.07,32.77,2.78,2.40,3.86,11.24,0.74,2.54,13.11,9.45
NE3,8.72,8.31,8.08,8.15,5.24,5.72,94.96,4.82,3.65,3.78,37.77,11.33,4.32,6.54,3.81
NO,27.06,20.36,24.18,24.01,24.92,5.56,45.73,3.10,2.69,12.06,17.68,2.25,11.92,5.72,10.50
NO2,2.72,9.90,14.08,9.41,6.63,2.71,14.59,3.83,1.76,4.51,10.80,19.85,2.43,3.93,7.08
NO3,18.60,28.80,28.70,28.72,16.42,12.93,86.88,9.87,9.87,38.64,14.68,1.78,9.91,8.84,2.07
NTE,22.71,24.79,24.29,24.33,23.06,8.48,15.64,20.81,17.14,58.45,28.11,9.43,19.69,18.08,20.50
NTE2,5.59,16.89,17.76,17.76,19.80,3.51,14.37,1.26,1.16,1.56,24.25,0.46,1.22,1.16,1.30



📊 REPORTE 3: RACHAS MÁXIMAS SIN DATOS (ESTACIÓN 'CE')
Para detectar si un sensor mide diario o estuvo apagado meses


,Contaminante,Max_Horas_Consecutivas_Vacias,Dias_Equivalentes
3,NOX,1333,55.5
0,CO,503,21.0
4,O3,501,20.9
6,PM2.5,432,18.0
2,NO2,294,12.2
1,NO,244,10.2
10,SO2,171,7.1
12,TOUT,76,3.2
7,PRS,76,3.2
8,RAINF,76,3.2
